In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast

In [2]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [3]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

/Users/elenanazarenko/opt/anaconda3/envs/py38/lib/python3.8/site-packages/spacy/util.py:877: UserWarning: [W095] Model 'de_core_news_sm' (3.0.0) was trained with spaCy v3.0 and may not be 100% compatible with the current version (3.5.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [4]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [5]:
def add_dots(row):
    list_out =[]
    if type(row)==str:
        row_list = row.split(" | ")
        for item in row_list:
            res = re.search(r"\...", item)
            if res is not None:
                list_out.append(item)
            else:
                list_out.append(item+"(...)")
        return " | ".join(list_out)

In [6]:
def clean_str(row):
    array_out =[]
    if len(row)==0:
        return list()
    else:
        array_out =[i.strip(' |') for i in row]
        
        return array_out

In [7]:
#function to split words and sentences into two different dataframes (currently doesn't used)
def split_words_sentences_short(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [8]:
#function to split words and sentences into two different dataframes for singular/plural dataframes
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [9]:
#function to separate one term binary pronouns excluding / (for example, exclude he/she)
def separate_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)
        
        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word
        
    


In [10]:
#function to split words and sentences into two different dataframes for singular/plural dataframes. Used for gendered category to handle multiterms pronouns
def split_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []
    
    list_secondary_subcategory_word = []
    list_secondary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
   
    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)
        
        
        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_secondary_subcategory_word.append(row["Secondary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        elif type(span) == re.Match and len(word.rsplit("/"))==2:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_secondary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory","Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_secondary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [11]:
def replace_tolist(file_csv):

    # creating a variable and storing the search text
    search_text = "['']"
  
    # creating a variable and storing the replace text
    replace_text = "[]"
  
    # Opening our text file in read only
    # mode using the open() function
    with open(file_csv, 'r') as file:
        
        # Reading the content of the file
        # using the read() function and storing
        # them in a new variable
        data = file.read()
  
        # Searching and replacing the text
        # using the replace() function
        data = data.replace(search_text, replace_text)
    
    with open(file_csv, 'w') as file:
        # Writing the replaced data in our
        # text file
        file.write(data)
    return file_csv



In [12]:
# read csv file
df_rules = pd.read_csv("rules_02MAR_German_h.csv")

In [13]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2714 entries, 0 to 2713
Data columns (total 18 columns):
Lemma                             2713 non-null object
Word_Type                         2714 non-null object
Priority                          25 non-null object
Category                          2714 non-null object
Primary_subcategory               2714 non-null object
Secondary_subcategory             788 non-null object
Alt_Field                         1049 non-null object
Alt_Sg_Replacement                1351 non-null object
Alt_Sg_/_and_inclusive_form       433 non-null object
Identity_first                    2 non-null object
Alt_Sg_people_first               290 non-null object
Alt_Sg_reframed                   258 non-null object
Alt_Pl_collective_noun            67 non-null object
Alt_Pl                            304 non-null object
Alt_Pl_pair_and_inclusive_form    360 non-null object
Identity_first_pl                 1 non-null object
Alt_Pl_people_first          

In [14]:
#drop row if all value in row are missing
df_rules.dropna(how='all', inplace=True)

In [15]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2714 entries, 0 to 2713
Data columns (total 18 columns):
Lemma                             2713 non-null object
Word_Type                         2714 non-null object
Priority                          25 non-null object
Category                          2714 non-null object
Primary_subcategory               2714 non-null object
Secondary_subcategory             788 non-null object
Alt_Field                         1049 non-null object
Alt_Sg_Replacement                1351 non-null object
Alt_Sg_/_and_inclusive_form       433 non-null object
Identity_first                    2 non-null object
Alt_Sg_people_first               290 non-null object
Alt_Sg_reframed                   258 non-null object
Alt_Pl_collective_noun            67 non-null object
Alt_Pl                            304 non-null object
Alt_Pl_pair_and_inclusive_form    360 non-null object
Identity_first_pl                 1 non-null object
Alt_Pl_people_first          

In [16]:
df_rules["Primary_subcategory"].unique()

array(['abbreviation', 'anglicism', 'titles', 'communal', 'sexism',
       'culture', 'ableism', 'd_and_i', 'exaggerating', 'age',
       'homophobia', 'transphobia', 'gender_identity', 'hidden_image',
       'agentic', 'function', 'hollow', 'offensive_language', 'migration',
       'classism', 'sports_terms', 'age_old', 'racist_source',
       'xenophobia', 'ability', 'belief', 'color', 'vision', 'leadership',
       'medical_state', 'cognitive_ability', 'male_stereotype', 'speech',
       'cognitive_perception', 'sexual_orientation', 'antisemitism',
       'racism', 'antimuslim', 'military_source', 'female_stereotype',
       'mental_wellbeing', 'age_young', 'mobility', 'formality', 'filler',
       'hearing', 'binary_pronouns', 'caring'], dtype=object)

### Drop duplicates

In [17]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Identity_first_pl,Alt_Pl_people_first,3rd_party_alternatives


In [18]:
#get index of duplicates
i1 = df_rules[((df_rules.Lemma == 'Ally'))].index

In [19]:
#get index of duplicates
i2 = df_rules[((df_rules.Lemma == 'LGBTQIA+'))].index

In [20]:
#get index of duplicates
i3 = df_rules[((df_rules.Lemma == 'Zwischengeschlechtlichkeit'))].index

In [21]:
#drop row with one duplicate
#df_rules = df_rules.drop(i1[1])

In [22]:
#drop row with one duplicate
#df_rules = df_rules.drop(i2[0])

In [23]:
#drop row with one duplicate
#df_rules = df_rules.drop(i3[0])

In [24]:
df_rules["Lemma"].duplicated().sum()

0

In [25]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2714 entries, 0 to 2713
Data columns (total 18 columns):
Lemma                             2713 non-null object
Word_Type                         2714 non-null object
Priority                          25 non-null object
Category                          2714 non-null object
Primary_subcategory               2714 non-null object
Secondary_subcategory             788 non-null object
Alt_Field                         1049 non-null object
Alt_Sg_Replacement                1351 non-null object
Alt_Sg_/_and_inclusive_form       433 non-null object
Identity_first                    2 non-null object
Alt_Sg_people_first               290 non-null object
Alt_Sg_reframed                   258 non-null object
Alt_Pl_collective_noun            67 non-null object
Alt_Pl                            304 non-null object
Alt_Pl_pair_and_inclusive_form    360 non-null object
Identity_first_pl                 1 non-null object
Alt_Pl_people_first          

In [26]:
df_rules[df_rules["Lemma"].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Identity_first_pl,Alt_Pl_people_first,3rd_party_alternatives
1122,NaN,acr,NaN,style,abbreviation,general,NaN,not applicable | not available | unzutreffend ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
#convert NaN to string "N/A" in lemmas
df_rules.Lemma.fillna("N/A", inplace=True)

In [28]:
df_rules[df_rules["Lemma"].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Identity_first_pl,Alt_Pl_people_first,3rd_party_alternatives


### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing

In [29]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Identity_first_pl,Alt_Pl_people_first,3rd_party_alternatives


In [30]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [31]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2714 entries, 0 to 2713
Data columns (total 18 columns):
Lemma                             2714 non-null object
Word_Type                         2714 non-null object
Priority                          25 non-null object
Category                          2714 non-null object
Primary_subcategory               2714 non-null object
Secondary_subcategory             788 non-null object
Alt_Field                         1049 non-null object
Alt_Sg_Replacement                1351 non-null object
Alt_Sg_/_and_inclusive_form       433 non-null object
Identity_first                    2 non-null object
Alt_Sg_people_first               290 non-null object
Alt_Sg_reframed                   258 non-null object
Alt_Pl_collective_noun            67 non-null object
Alt_Pl                            304 non-null object
Alt_Pl_pair_and_inclusive_form    360 non-null object
Identity_first_pl                 1 non-null object
Alt_Pl_people_first          

### add (...) to 
Refraimed columns:
Alt_Pl_people_first
Alt_Pl_reframed 

Alt_Sg_people_first
Alt_Sg_reframed

In [32]:
# add (...) for inspiration alternatives
#df_rules["Alt_Pl_reframed"]=df_rules["Alt_Pl_reframed"].map(add_dots)

In [33]:
# add (...) for inspiration alternatives
df_rules["Alt_Pl_people_first"]=df_rules["Alt_Pl_people_first"].map(add_dots)

In [34]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_people_first"]=df_rules["Alt_Sg_people_first"].map(add_dots)

In [35]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_reframed"]=df_rules["Alt_Sg_reframed"].map(add_dots)

## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [36]:
#order for plural form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Identity_first", "Alt_Sg_Replacement", "Alt_Field", "Alt_Sg_/_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [37]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [38]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Identity_first_pl", "Alt_Pl", "Alt_Pl_pair_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [39]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [40]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [41]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives'] = df_rules.iloc[:,source_col_loc:source_col_loc+11].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)


In [42]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [43]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,Alt_Pl_pair_and_inclusive_form,Identity_first_pl,Alt_Pl_people_first,3rd_party_alternatives,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service ((x)aaS) | etwas als Di...,NaN,NaN,...,NaN,NaN,None,NaN,something as a service ((x)aaS) | etwas als Di...,"[something as a service ((x)aaS), etwas als Di...",,[],something as a service ((x)aaS) | etwas als Di...,"[something as a service ((x)aaS), etwas als Di..."
1,ACCT,abbr,NaN,style,abbreviation,finance,NaN,Account | Konto,NaN,NaN,...,NaN,NaN,None,NaN,Account | Konto,"[Account, Konto]",,[],Account | Konto,"[Account, Konto]"
2,ADD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Disorder | Attention Deficit...,NaN,NaN,...,NaN,NaN,None,NaN,Attention Deficit Disorder | Attention Deficit...,"[Attention Deficit Disorder, Attention Deficit...",,[],Attention Deficit Disorder | Attention Deficit...,"[Attention Deficit Disorder, Attention Deficit..."
3,ADHD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Hyperactivity Disorder | Att...,NaN,NaN,...,NaN,NaN,None,NaN,Attention Deficit Hyperactivity Disorder | Att...,"[Attention Deficit Hyperactivity Disorder, Att...",,[],Attention Deficit Hyperactivity Disorder | Att...,"[Attention Deficit Hyperactivity Disorder, Att..."
4,ADHS,acr,NaN,style,abbreviation,medical,NaN,Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,NaN,NaN,...,NaN,NaN,None,NaN,Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,[Aufmerksamkeitsdefizit-Hyperaktivitätsstörung...,,[],Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,[Aufmerksamkeitsdefizit-Hyperaktivitätsstörung...


### cleaning alternatives

In [44]:
# strip alternatives 
df_rules["Alt_split"]=df_rules["Alt_split"].map(clean_str)

In [45]:
# strip alternatives 
df_rules["Sg_all_split"]=df_rules["Sg_all_split"].map(clean_str)

In [46]:
# strip alternatives 
df_rules["Pl_all_split"]=df_rules["Pl_all_split"].map(clean_str)

### Create subsets of dataframes

In [47]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"]]

### Check duplicates&homonyms

In [48]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


## Binary_pronouns

In [49]:
df_binary_pronouns = df_rules[df_rules["Primary_subcategory"]=="binary_pronouns"]

In [50]:
df_binary_pronouns.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16 entries, 2010 to 2490
Data columns (total 8 columns):
Lemma                  16 non-null object
Word_Type              16 non-null object
Category               16 non-null object
Primary_subcategory    16 non-null object
Sg_all_split           16 non-null object
Pl_all                 16 non-null object
Pl_all_split           16 non-null object
Alt_split              16 non-null object
dtypes: object(8)
memory usage: 1.1+ KB


In [51]:
df_binary_pronouns

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
2010,er,s,gendered,binary_pronouns,"[du, Sie, ((Vorname)), ((Vorname Nachname)), s...",,[],"[du, Sie, ((Vorname)), ((Vorname Nachname)), s..."
2163,ihm,s,gendered,binary_pronouns,"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))...",,[],"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))..."
2164,ihn,s,gendered,binary_pronouns,"[dich, Sie, ((Vorname)), ((Vorname Nachname)),...",,[],"[dich, Sie, ((Vorname)), ((Vorname Nachname)),..."
2165,ihr,s,gendered,binary_pronouns,"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo...",,[],"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo..."
2166,ihre,s,gendered,binary_pronouns,"[deine, Ihre, ((Vorname))s, ((Vorname Nachname...",,[],"[deine, Ihre, ((Vorname))s, ((Vorname Nachname..."
2168,ihrem,s,gendered,binary_pronouns,"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna...",,[],"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna..."
2169,ihren,s,gendered,binary_pronouns,"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna...",,[],"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna..."
2170,ihrer,s,gendered,binary_pronouns,"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna...",,[],"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna..."
2171,ihres,s,gendered,binary_pronouns,"[deines, Ihres, ((Vorname))s, ((Vorname Nachna...",,[],"[deines, Ihres, ((Vorname))s, ((Vorname Nachna..."
2464,sein,s,gendered,binary_pronouns,"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))...",,[],"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))..."


In [52]:
# separate one term binary pronouns 
#df_binary_pronouns_word =separate_binary_pronouns(df_binary_pronouns)

In [53]:
#check the results
#df_binary_pronouns_word

In [54]:
#create subset of dataframe to drop unnecessary columns
df_binary_pronouns = df_binary_pronouns[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all_split", "Alt_split"]]

In [55]:
#save one term binary_pronouns to csv file after preprocessing
df_binary_pronouns.to_csv("gendered_singular_they.csv", index=False)

In [56]:
df_binary_pronouns.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16 entries, 2010 to 2490
Data columns (total 7 columns):
Lemma                  16 non-null object
Word_Type              16 non-null object
Category               16 non-null object
Primary_subcategory    16 non-null object
Sg_all_split           16 non-null object
Pl_all_split           16 non-null object
Alt_split              16 non-null object
dtypes: object(7)
memory usage: 1.0+ KB


In [57]:
df_binary_pronouns

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split
2010,er,s,gendered,binary_pronouns,"[du, Sie, ((Vorname)), ((Vorname Nachname)), s...",[],"[du, Sie, ((Vorname)), ((Vorname Nachname)), s..."
2163,ihm,s,gendered,binary_pronouns,"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))...",[],"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))..."
2164,ihn,s,gendered,binary_pronouns,"[dich, Sie, ((Vorname)), ((Vorname Nachname)),...",[],"[dich, Sie, ((Vorname)), ((Vorname Nachname)),..."
2165,ihr,s,gendered,binary_pronouns,"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo...",[],"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo..."
2166,ihre,s,gendered,binary_pronouns,"[deine, Ihre, ((Vorname))s, ((Vorname Nachname...",[],"[deine, Ihre, ((Vorname))s, ((Vorname Nachname..."
2168,ihrem,s,gendered,binary_pronouns,"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna...",[],"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna..."
2169,ihren,s,gendered,binary_pronouns,"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna...",[],"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna..."
2170,ihrer,s,gendered,binary_pronouns,"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna...",[],"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna..."
2171,ihres,s,gendered,binary_pronouns,"[deines, Ihres, ((Vorname))s, ((Vorname Nachna...",[],"[deines, Ihres, ((Vorname))s, ((Vorname Nachna..."
2464,sein,s,gendered,binary_pronouns,"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))...",[],"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))..."


In [58]:
replace_tolist(r"gendered_singular_they.csv")

'gendered_singular_they.csv'

In [59]:
#test read of dataframe
df_test = pd.read_csv("gendered_singular_they.csv")

In [60]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 7 columns):
Lemma                  16 non-null object
Word_Type              16 non-null object
Category               16 non-null object
Primary_subcategory    16 non-null object
Sg_all_split           16 non-null object
Pl_all_split           16 non-null object
Alt_split              16 non-null object
dtypes: object(7)
memory usage: 1.0+ KB


In [61]:
df_test

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split
0,er,s,gendered,binary_pronouns,"['du', 'Sie', '((Vorname))', '((Vorname Nachna...",[],"['du', 'Sie', '((Vorname))', '((Vorname Nachna..."
1,ihm,s,gendered,binary_pronouns,"['dir', 'Ihnen', '((Vorname))', '((Vorname Nac...",[],"['dir', 'Ihnen', '((Vorname))', '((Vorname Nac..."
2,ihn,s,gendered,binary_pronouns,"['dich', 'Sie', '((Vorname))', '((Vorname Nach...",[],"['dich', 'Sie', '((Vorname))', '((Vorname Nach..."
3,ihr,s,gendered,binary_pronouns,"['dir', 'dein', 'Ihnen', 'Ihr', 'Ihr', '((Vorn...",[],"['dir', 'dein', 'Ihnen', 'Ihr', 'Ihr', '((Vorn..."
4,ihre,s,gendered,binary_pronouns,"['deine', 'Ihre', '((Vorname))s', '((Vorname N...",[],"['deine', 'Ihre', '((Vorname))s', '((Vorname N..."
5,ihrem,s,gendered,binary_pronouns,"['deinem', 'Ihrem', '((Vorname))s', '((Vorname...",[],"['deinem', 'Ihrem', '((Vorname))s', '((Vorname..."
6,ihren,s,gendered,binary_pronouns,"['deinen', 'Ihren', '((Vorname))s', '((Vorname...",[],"['deinen', 'Ihren', '((Vorname))s', '((Vorname..."
7,ihrer,s,gendered,binary_pronouns,"['deiner', 'Ihrer', '((Vorname))s', '((Vorname...",[],"['deiner', 'Ihrer', '((Vorname))s', '((Vorname..."
8,ihres,s,gendered,binary_pronouns,"['deines', 'Ihres', '((Vorname))s', '((Vorname...",[],"['deines', 'Ihres', '((Vorname))s', '((Vorname..."
9,sein,s,gendered,binary_pronouns,"['dein', 'Ihr', '((Vorname))s', '((Vorname Nac...",[],"['dein', 'Ihr', '((Vorname))s', '((Vorname Nac..."


### Remove one term binary_pronouns from dataframe for the next steps pre-processing

In [62]:
#remove binary pronouns
binary_list = list(df_binary_pronouns["Lemma"])

In [63]:
len(binary_list)

16

In [64]:
df_rules = df_rules.query("Lemma != @binary_list")

In [65]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2698 entries, 0 to 2713
Data columns (total 8 columns):
Lemma                  2698 non-null object
Word_Type              2698 non-null object
Category               2698 non-null object
Primary_subcategory    2698 non-null object
Sg_all_split           2698 non-null object
Pl_all                 2698 non-null object
Pl_all_split           2698 non-null object
Alt_split              2698 non-null object
dtypes: object(8)
memory usage: 189.7+ KB


In [66]:
df_nazi = df_rules[df_rules["Primary_subcategory"]=="Nazi_language"]
df_nazi

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


## Acronyms (abbreviations)

In [67]:
df_rules["Word_Type"].unique()

array(['acr', 'abbr', 's', 'i', 's+v', 'a+s', '=a+s', '=s', '-acr', 'a',
       'adv+a', 'adv', 'v', '=a', 'a+v'], dtype=object)

In [68]:
df_abbr = df_rules[(df_rules["Word_Type"]=='abbr') | (df_rules["Word_Type"]=='acr')| (df_rules["Word_Type"]=='-acr')]

In [69]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 307 entries, 0 to 2554
Data columns (total 8 columns):
Lemma                  307 non-null object
Word_Type              307 non-null object
Category               307 non-null object
Primary_subcategory    307 non-null object
Sg_all_split           307 non-null object
Pl_all                 307 non-null object
Pl_all_split           307 non-null object
Alt_split              307 non-null object
dtypes: object(8)
memory usage: 21.6+ KB


In [70]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [71]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations.csv", index=False)

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [72]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [73]:
df_rules = df_rules.query("Lemma != @abbr_list")

In [74]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2391 entries, 20 to 2713
Data columns (total 8 columns):
Lemma                  2391 non-null object
Word_Type              2391 non-null object
Category               2391 non-null object
Primary_subcategory    2391 non-null object
Sg_all_split           2391 non-null object
Pl_all                 2391 non-null object
Pl_all_split           2391 non-null object
Alt_split              2391 non-null object
dtypes: object(8)
memory usage: 168.1+ KB


## Openly_discriminating

In [75]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [76]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 425 entries, 23 to 2646
Data columns (total 8 columns):
Lemma                  425 non-null object
Word_Type              425 non-null object
Category               425 non-null object
Primary_subcategory    425 non-null object
Sg_all_split           425 non-null object
Pl_all                 425 non-null object
Pl_all_split           425 non-null object
Alt_split              425 non-null object
dtypes: object(8)
memory usage: 29.9+ KB


In [77]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
23,Abwichshure,s,openly_discriminating,sexism,[-],-,[-],[-]
29,Affenfotze,s,openly_discriminating,sexism,[-],-,[-],[-]
32,Albino,s,openly_discriminating,ableism,[-],-,[-],[-]
33,Allerweltshure,s,openly_discriminating,sexism,[-],-,[-],[-]
35,Allmannshure,s,openly_discriminating,sexism,[-],-,[-],[-]


In [78]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [79]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Abwichshure,s,openly_discriminating,sexism,[-],-,[-],[-]
1,Affenfotze,s,openly_discriminating,sexism,[-],-,[-],[-]
2,Albino,s,openly_discriminating,ableism,[-],-,[-],[-]
3,Allerweltshure,s,openly_discriminating,sexism,[-],-,[-],[-]
4,Allmannshure,s,openly_discriminating,sexism,[-],-,[-],[-]


In [80]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [81]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words.csv", index=False)

In [82]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences.csv", index=False)

## Style

In [83]:
df_style = df_rules[df_rules["Category"]=="style"]

In [84]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 576 entries, 24 to 2713
Data columns (total 8 columns):
Lemma                  576 non-null object
Word_Type              576 non-null object
Category               576 non-null object
Primary_subcategory    576 non-null object
Sg_all_split           576 non-null object
Pl_all                 576 non-null object
Pl_all_split           576 non-null object
Alt_split              576 non-null object
dtypes: object(8)
memory usage: 40.5+ KB


In [85]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [86]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [87]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words.csv", index=False)

In [88]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences.csv", index=False)

## Inclusive

In [89]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [90]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 332 entries, 22 to 2694
Data columns (total 8 columns):
Lemma                  332 non-null object
Word_Type              332 non-null object
Category               332 non-null object
Primary_subcategory    332 non-null object
Sg_all_split           332 non-null object
Pl_all                 332 non-null object
Pl_all_split           332 non-null object
Alt_split              332 non-null object
dtypes: object(8)
memory usage: 23.3+ KB


In [91]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
22,Abhängigkeit,s,inclusive,communal,[],,[],[]
26,Achtsamkeit,s,inclusive,communal,[],,[],[]
34,Allies,s,inclusive,d_and_i,[Unterstützer~in~/~Unterstützer],Unterstützer~innen~ und ~Unterstützer,[Unterstützer~innen~ und ~Unterstützer],"[Unterstützer~in~/~Unterstützer, Unterstützer~..."
37,Allonormativität,s,inclusive,d_and_i,[],,[],[]
38,Ally,s,inclusive,d_and_i,"[Unterstützung, Unterstützer~in~/~Unterstützer...",Unterstützende | Unterstützer~innen~ und ~Unte...,"[Unterstützende, Unterstützer~innen~ und ~Unte...","[Unterstützung, Unterstützer~in~/~Unterstützer..."


In [92]:
df_inclusive["Primary_subcategory"].unique()

array(['communal', 'd_and_i', 'caring'], dtype=object)

In [93]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [94]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [95]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 4 columns):
Lemma                  18 non-null object
Word_Type              18 non-null object
Category               18 non-null object
Primary_subcategory    18 non-null object
dtypes: object(4)
memory usage: 704.0+ bytes


In [96]:
df_inclusive_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Data columns (total 4 columns):
Lemma                  314 non-null object
Word_Type              314 non-null object
Category               314 non-null object
Primary_subcategory    314 non-null object
dtypes: object(4)
memory usage: 9.9+ KB


In [97]:
df_communal_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='communal']

In [98]:
df_communal_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 220 entries, 0 to 313
Data columns (total 4 columns):
Lemma                  220 non-null object
Word_Type              220 non-null object
Category               220 non-null object
Primary_subcategory    220 non-null object
dtypes: object(4)
memory usage: 8.6+ KB


In [99]:
df_communal_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='communal']

In [100]:
df_communal_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 4 columns):
Lemma                  0 non-null object
Word_Type              0 non-null object
Category               0 non-null object
Primary_subcategory    0 non-null object
dtypes: object(4)
memory usage: 0.0+ bytes


In [101]:
#save to csv file after preprocessing
df_communal_word.to_csv("communal.csv", index=False)

In [102]:
df_d_and_i_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='d_and_i']

In [103]:
df_d_and_i_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 93 entries, 2 to 296
Data columns (total 4 columns):
Lemma                  93 non-null object
Word_Type              93 non-null object
Category               93 non-null object
Primary_subcategory    93 non-null object
dtypes: object(4)
memory usage: 3.6+ KB


In [104]:
df_d_and_i_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='d_and_i']

In [105]:
df_d_and_i_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 18 entries, 0 to 17
Data columns (total 4 columns):
Lemma                  18 non-null object
Word_Type              18 non-null object
Category               18 non-null object
Primary_subcategory    18 non-null object
dtypes: object(4)
memory usage: 720.0+ bytes


In [106]:
#save to csv file after preprocessing
df_d_and_i_word.to_csv("d_and_i_words.csv", index=False)

In [107]:
#save to csv file after preprocessing
df_d_and_i_sentence.to_csv("d_and_i_sentences.csv", index=False)

## Gendered

In [108]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [109]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 345 entries, 20 to 2698
Data columns (total 8 columns):
Lemma                  345 non-null object
Word_Type              345 non-null object
Category               345 non-null object
Primary_subcategory    345 non-null object
Sg_all_split           345 non-null object
Pl_all                 345 non-null object
Pl_all_split           345 non-null object
Alt_split              345 non-null object
dtypes: object(8)
memory usage: 24.3+ KB


In [110]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_words_sentences(df_gendered)

In [111]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 323 entries, 0 to 322
Data columns (total 8 columns):
Lemma                  323 non-null object
Word_Type              323 non-null object
Category               323 non-null object
Primary_subcategory    323 non-null object
Sg_all_split           323 non-null object
Pl_all                 323 non-null object
Pl_all_split           323 non-null object
Alt_split              323 non-null object
dtypes: object(8)
memory usage: 20.3+ KB


In [112]:
df_gendered_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 8 columns):
Lemma                  22 non-null object
Word_Type              22 non-null object
Category               22 non-null object
Primary_subcategory    22 non-null object
Sg_all_split           22 non-null object
Pl_all                 22 non-null object
Pl_all_split           22 non-null object
Alt_split              22 non-null object
dtypes: object(8)
memory usage: 1.5+ KB


In [113]:
df_gendered_word.query("Word_Type == '-s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [114]:
df_gendered_word.query("Word_Type == '=s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [115]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's'")

In [116]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 315 entries, 0 to 318
Data columns (total 8 columns):
Lemma                  315 non-null object
Word_Type              315 non-null object
Category               315 non-null object
Primary_subcategory    315 non-null object
Sg_all_split           315 non-null object
Pl_all                 315 non-null object
Pl_all_split           315 non-null object
Alt_split              315 non-null object
dtypes: object(8)
memory usage: 22.1+ KB


In [117]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [118]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all"], axis=1)

In [119]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words.csv", index=False)

In [120]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words.csv", index=False)

In [121]:
df_gendered_sentence.to_csv("gendered_sentences.csv", index=False)

## Unconscious Bias

In [122]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [123]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 713 entries, 30 to 2712
Data columns (total 8 columns):
Lemma                  713 non-null object
Word_Type              713 non-null object
Category               713 non-null object
Primary_subcategory    713 non-null object
Sg_all_split           713 non-null object
Pl_all                 713 non-null object
Pl_all_split           713 non-null object
Alt_split              713 non-null object
dtypes: object(8)
memory usage: 50.1+ KB


In [124]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences(df_ub)

In [125]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566 entries, 0 to 565
Data columns (total 8 columns):
Lemma                  566 non-null object
Word_Type              566 non-null object
Category               566 non-null object
Primary_subcategory    566 non-null object
Sg_all_split           566 non-null object
Pl_all                 566 non-null object
Pl_all_split           566 non-null object
Alt_split              566 non-null object
dtypes: object(8)
memory usage: 35.5+ KB


In [126]:
df_ub_word[df_ub_word["Word_Type"]=='-s']

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [127]:
df_ub_word[df_ub_word["Word_Type"]=='=s']

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
102,Farbige,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC ...,"[Personen, Menschen, People of Color, BIPoC, -...","[--- nicht auf Menschen beziehen, Person, Mens..."


In [128]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [129]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 143 entries, 0 to 326
Data columns (total 8 columns):
Lemma                  143 non-null object
Word_Type              143 non-null object
Category               143 non-null object
Primary_subcategory    143 non-null object
Sg_all_split           143 non-null object
Pl_all                 143 non-null object
Pl_all_split           143 non-null object
Alt_split              143 non-null object
dtypes: object(8)
memory usage: 10.1+ KB


#### save the left data into separate dataframe

In [130]:
plural_list = list(df_ub_word_pl["Lemma"])

In [131]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all" 

In [132]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all"], axis=1)

#### save to csv files after preprocessing

In [133]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words.csv", index=False)
df_ub_sentence.to_csv("ub_sentences.csv", index=False)